# Extraction accuracy against Ground truth (#19)

This notebook is the accuracy test of record for document reading (spec #1). It reads every document of the
test set through the same `extract_document()` path the extractor uses (docling-serve OCR on CPU, then
gpt-oss-20b with a JSON schema). It compares four fields with the Ground truth and logs the result to MLflow
as one evaluation run in experiment `ordo-evaluation`.

**Test set**
- The 47 November documents of the Borsuk Labs scenario (`sim/documents/november/`): 25 text PDFs and
  22 rendered phone photos with defects (blur, glare, crumpled, shadow, perspective...).
- Real phone photos of the printed SAMPLE receipts (`notebooks/receipt_photos/<truth id>-<n>.jpg`,
  EXIF stripped by `notebooks/strip_exif.py`).

**Ground truth**: `sim/scenario/out/november_documents.json` for the November documents. Role 4's
`ground_truth.json` lists the same 47 ids, but only with gross and date; this file is its per-document
source with all four fields. For the receipts: `sim/documents/receipts/receipts_truth.json`.

**Counting rules** (`backend/pipeline/extract/evaluation.py`, unit-tested)
- A field is right when the value read equals the Ground truth, whatever the Document's status.
- A field is **Silently wrong** when it is wrong and the Document was accepted without review
  (status `extracted`). This is the number that must stay at zero.
- Gross: amount to the cent and currency. Date: the issue date. Seller NIP: compared without spaces,
  dashes, a printed label (NIP, EIN, VAT) or the PL prefix. VAT: total VAT; a VAT that was not read
  counts as equal to a Ground truth of 0.00.

**Run it**: `deploy/platform/setup.sh eval-workbench` starts workbench `model-tests` with the MLflow
settings, then Run All here. Rows run one at a time (`MLFLOW_GENAI_EVAL_MAX_WORKERS=1`), so the
timings are real: about 6 s per document. Afterwards: `deploy/platform/setup.sh stop-workbench`
(the workbench never runs during the demo).

In [ ]:
# The cluster's MLflow server is 3.14; the client matches it, as the backend's mlflow-tracing does (#18).
# The workbench's default index is Red Hat's RHOAI mirror, which lacks these versions; PyPI as in extract_bench.sh.
# A workbench restart resets the image's packages, so this cell runs every time.
%pip install -q --index-url https://pypi.org/simple "mlflow>=3.14,<3.15" "openai==3.57.0" "httpx==0.28.1" "jsonschema==4.26.0"

In [ ]:
import os
import re
import subprocess
import sys
from dataclasses import asdict
from datetime import date
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_WORKERS", "1")
os.environ.setdefault("MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION", "true")  # else MLflow reads row 1 twice, untraced

EXPERIMENT = "ordo-evaluation"
AS_OF = date(2026, 11, 21)  # Borsuk Labs' as-of date: the date checks see the documents as the demo does
NOVEMBER_TRUTH = REPO / "sim/scenario/out/november_documents.json"
DOCUMENTS = REPO / "sim/documents"
RECEIPT_TRUTH = REPO / "sim/documents/receipts/receipts_truth.json"
PHOTOS = REPO / "notebooks/receipt_photos"
MLFLOW_UI = "https://rh-ai.apps.ocp.w6k4q.sandbox2219.opentlc.com/mlflow"

needed = ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_TOKEN", "MLFLOW_WORKSPACE", "LLM_BASE_URL", "ORDO_DOCLING_URL"]
missing = [name for name in needed if not os.environ.get(name)]
assert not missing, f"missing {missing}: run deploy/platform/setup.sh eval-workbench"
print(f"repo {REPO}, as-of {AS_OF}, rows in parallel: {os.environ['MLFLOW_GENAI_EVAL_MAX_WORKERS']}")

In [ ]:
import mlflow
import pandas as pd
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

from backend import settings, tracing
from backend.pipeline.extract import docling, evaluation, model
from backend.pipeline.extract.evaluation import FIELDS, SUBSETS, Truth

assert tracing.setup(EXPERIMENT), "MLflow is not usable; the 'tracing off' warning above says why"
GIT_SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
DOCLING_IMAGE = re.search(r"image:\s*(\S*docling\S*)", (REPO / "deploy/models/docling-serve.yaml").read_text()).group(1)
print(f"mlflow {mlflow.__version__}, experiment {EXPERIMENT} in workspace {os.environ['MLFLOW_WORKSPACE']}")
print(f"commit {GIT_SHA[:7]}, model {settings.llm_model()} at {settings.llm_base_url()}")
print(f"docling {DOCLING_IMAGE} at {os.environ['ORDO_DOCLING_URL']}")

In [ ]:
# One client each for the whole run, as the extractor keeps them.
docling_client = docling.default_client()
llm_client = model.default_client()

In [ ]:
cases = evaluation.load_november(NOVEMBER_TRUTH, DOCUMENTS) + evaluation.load_receipt_photos(RECEIPT_TRUTH, PHOTOS)
CASES = {case.case_id: case for case in cases}
COUNTS = {subset: sum(case.subset == subset for case in cases) for subset in SUBSETS}
assert len(CASES) == len(cases), "two documents share an id"
pd.Series(COUNTS, name="documents").to_frame()

## Predict function and scorers

MLflow calls `read_document` once per row and traces it. Inside, `extract_document`'s own spans
(`docling.convert`, the gpt-oss chat call, `build_document`) nest under the row's trace, so each row in
the Evaluations tab opens to what docling saw and what the model answered. The scorers are thin wrappers
over `evaluation.compare`; each one gives its verdict with a "read X, truth Y" rationale.

In [ ]:
OUTPUTS = {}


@mlflow.trace(name="read_document")
def read_document(case_id: str, path: str, subset: str) -> dict:
    tracing.tag(case_id=case_id, subset=subset)
    read = evaluation.read_case(Path(path), today=AS_OF, docling_client=docling_client, llm_client=llm_client)
    OUTPUTS[case_id] = read
    return read


def field_feedback(field: str, outputs: dict, expectations: dict) -> Feedback:
    truth = Truth(**expectations)
    right = evaluation.compare(outputs, truth)[field]
    why = outputs["error"] or f"read {evaluation.read_text(outputs, field)}, truth {evaluation.truth_text(truth, field)}"
    return Feedback(value=right, rationale=why)


@scorer
def gross_correct(outputs: dict, expectations: dict) -> Feedback:
    return field_feedback("gross", outputs, expectations)


@scorer
def date_correct(outputs: dict, expectations: dict) -> Feedback:
    return field_feedback("issue_date", outputs, expectations)


@scorer
def seller_nip_correct(outputs: dict, expectations: dict) -> Feedback:
    return field_feedback("seller_tax_id", outputs, expectations)


@scorer
def vat_correct(outputs: dict, expectations: dict) -> Feedback:
    return field_feedback("vat", outputs, expectations)


@scorer
def silently_wrong(outputs: dict, expectations: dict) -> Feedback:
    truth = Truth(**expectations)
    fields = evaluation.silently_wrong(outputs, evaluation.compare(outputs, truth))
    why = (f"accepted without review but wrong: {', '.join(evaluation.FIELD_NAMES[f] for f in fields)}"
           if fields else "nothing wrong was accepted without review")
    return Feedback(value=bool(fields), rationale=why)


@scorer
def needs_review(outputs: dict) -> Feedback:
    return Feedback(value=outputs["status"] == "needs_review",
                    rationale=f"status {outputs['status']}, confidence {outputs['confidence']}")


SCORERS = [gross_correct, date_correct, seller_nip_correct, vat_correct, silently_wrong, needs_review]

In [ ]:
data = [{"inputs": {"case_id": case.case_id, "path": str(case.path.relative_to(REPO)), "subset": case.subset},
         "expectations": asdict(case.truth)} for case in cases]

with mlflow.start_run(run_name=f"extraction-eval {GIT_SHA[:7]}") as run:
    mlflow.log_params({"git_sha": GIT_SHA, "model": settings.llm_model(), "docling_image": DOCLING_IMAGE,
                       "as_of": AS_OF.isoformat(), "documents": len(cases),
                       **{f"documents_{subset}": count for subset, count in COUNTS.items()}})
    results = mlflow.genai.evaluate(data=data, predict_fn=read_document, scorers=SCORERS)

    rows = [evaluation.score(case, OUTPUTS[case.case_id]) for case in cases]
    metrics = evaluation.summarise(rows)
    mlflow.log_metrics(metrics)
    table = pd.DataFrame(rows).assign(silently_wrong=lambda t: t.silently_wrong.str.join(", "))
    mlflow.log_table(table, artifact_file="per_document.json")
    HEADLINE = evaluation.headline(rows)
    mlflow.set_tag("headline", HEADLINE.splitlines()[0])
    mlflow.log_text(HEADLINE, "headline.txt")

docling_client.close()
{name: value for name, value in sorted(results.metrics.items())}

## Per document

Every document with its verdicts. Below it: only the documents with at least one wrong field, with what
was read next to the Ground truth.

In [ ]:
ok = [f"{field}_ok" for field in FIELDS]
pd.set_option("display.max_rows", 200)
table[["case_id", "subset", "status", "confidence", *ok, "silently_wrong", "total_seconds", "error"]]

In [ ]:
read_vs_truth = [column for field in FIELDS for column in (f"{field}_read", f"{field}_truth")]
table.loc[~table[ok].all(axis=1), ["case_id", "status", "silently_wrong", *read_vs_truth]]

In [ ]:
scopes = [scope for scope in ("all", *SUBSETS) if f"{scope}/documents" in metrics]
pd.DataFrame({scope: {key.split("/", 1)[1]: value for key, value in metrics.items() if key.startswith(f"{scope}/")}
              for scope in scopes})

## Headline for the pitch

Posted as a comment on #19.

In [ ]:
run_url = f"{MLFLOW_UI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"
print(f"""**Extraction accuracy: evaluation run of record**

{HEADLINE}

- Test set: {', '.join(f'{count} {subset.replace("_", " ")}' for subset, count in COUNTS.items() if count)}
- MLflow: experiment `{EXPERIMENT}` (workspace `{os.environ['MLFLOW_WORKSPACE']}`), run `{run.info.run_id}`: {run_url}
- Commit {GIT_SHA[:7]}, model {settings.llm_model()}, docling `{DOCLING_IMAGE}`, as-of {AS_OF}, one document at a time
- Mean {metrics['all/total_seconds_mean']} s per document, max {metrics['all/total_seconds_max']} s, {metrics['all/over_budget']} over 10 s""")